In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Paths
data_root = "/content/drive/MyDrive/processed_data"
train_dir = data_root + "/train"
val_dir = data_root + "/test"
save_path = "/content/drive/MyDrive/Model/model_cnn.pth"

os.makedirs("/content/drive/MyDrive/Model", exist_ok=True)

# -------------------------------
# Data Transform (Augmentation)
# -------------------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize(150),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
])

# Datasets and Loaders
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)

# -------------------------------
# Custom CNN Model
# -------------------------------
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = CNNModel(num_classes).to(device)

# -------------------------------
# Loss, Optimizer, Regularization
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)  # L2 regularization

# -------------------------------
# Training with Early Stopping
# -------------------------------
best_acc = 0.0
patience = 7
no_improve = 0
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        no_improve = 0
        torch.save(model.state_dict(), save_path)
        print("Model improved! Saved.")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping triggered!")
            break

print("Training complete. Best Accuracy:", best_acc)


Mounted at /content/drive
Using: cuda
Epoch 1/50 | Loss: 46.6001 | Val Acc: 0.6200
Model improved! Saved.
Epoch 2/50 | Loss: 32.4477 | Val Acc: 0.6400
Model improved! Saved.
Epoch 3/50 | Loss: 32.1555 | Val Acc: 0.6925
Model improved! Saved.
Epoch 4/50 | Loss: 32.0260 | Val Acc: 0.6725
Epoch 5/50 | Loss: 31.1827 | Val Acc: 0.6925
Epoch 6/50 | Loss: 30.5841 | Val Acc: 0.7175
Model improved! Saved.
Epoch 7/50 | Loss: 30.7916 | Val Acc: 0.7350
Model improved! Saved.
Epoch 8/50 | Loss: 30.3675 | Val Acc: 0.7300
Epoch 9/50 | Loss: 30.6720 | Val Acc: 0.7450
Model improved! Saved.
Epoch 10/50 | Loss: 30.3273 | Val Acc: 0.7350
Epoch 11/50 | Loss: 28.9536 | Val Acc: 0.7575
Model improved! Saved.
Epoch 12/50 | Loss: 30.0992 | Val Acc: 0.7650
Model improved! Saved.
Epoch 13/50 | Loss: 29.4073 | Val Acc: 0.7500
Epoch 14/50 | Loss: 29.5330 | Val Acc: 0.7700
Model improved! Saved.
Epoch 15/50 | Loss: 29.1437 | Val Acc: 0.7400
Epoch 16/50 | Loss: 29.2957 | Val Acc: 0.7625
Epoch 17/50 | Loss: 28.9445 